# OLMo 3 7B: prompts congelados e memória
Após instalar dependências, reinicie a sessão antes de carregar o modelo. Execute uma variante por arquivo.
Entrada: lista com `id`, `expected_answer` e `prompt` (mensagens system/user prontas). A resposta esperada nunca é enviada ao modelo.
O template de chat é aplicado uma vez pelo tokenizer do modelo. Não há nova seleção ou truncamento de chunks.
Cache offloaded usa RAM da CPU e pode reduzir a velocidade. Limpeza de cache libera reservas não utilizadas, mas não elimina o pico necessário à inferência.
O resultado com 7B quantizado é uma avaliação exploratória; não equivale à competição com OLMo 3.1 32B. Não foi validado em T4 neste repositório.


In [ ]:
%pip install -q "transformers==5.17.0" accelerate bitsandbytes


In [ ]:
import gc
import hashlib
import json
from pathlib import Path
from importlib.metadata import version

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

INPUT_JSON = Path("/content/select-hybrid-order-ce100-query-preserved.json")
OUTPUT_JSON = INPUT_JSON.with_name(INPUT_JSON.stem + "_answers.json")
MODEL_ID = "allenai/Olmo-3-7B-Instruct"
MAX_NEW_TOKENS = 2048  # Reduzido ligeiramente de 2048 para prevenir picos de contexto longo, mas ainda suficiente para a maioria das respostas.
CACHE_IMPLEMENTATION = None  # Desativado o 'offloaded' para evitar overheads e picos instáveis de alocação de VRAM.

payload = json.loads(INPUT_JSON.read_text(encoding="utf-8"))
assert isinstance(payload, list)
assert len({str(q["id"]) for q in payload}) == len(payload)
for q in payload:
    assert set(q) == {"id", "expected_answer", "prompt"}
    assert [m["role"] for m in q["prompt"]] == ["system", "user"]
    assert all(isinstance(m["content"], str) for m in q["prompt"])
assert torch.cuda.is_available(), "Select a GPU runtime in Colab"

In [ ]:
# Re-running this cell releases the previous model first.
if "model" in globals():
    del model
gc.collect()
torch.cuda.empty_cache()

# Força float16 explicitamente, pois a GPU T4 do Colab não suporta bfloat16 nativamente (o que causa picos ocultos de VRAM e OOM)
compute_dtype = torch.float16

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",  # Deixa o Accelerate gerenciar de forma ideal
    quantization_config=quantization_config,
    torch_dtype=compute_dtype,
    attn_implementation="sdpa",
)
model.eval()
print(torch.cuda.get_device_name(), "allocated GiB:", torch.cuda.memory_allocated()/2**30)

In [ ]:
metadata = {
    "model_id": MODEL_ID,
    "input_sha256": hashlib.sha256(INPUT_JSON.read_bytes()).hexdigest(),
    "max_new_tokens": MAX_NEW_TOKENS,
    "cache_implementation": CACHE_IMPLEMENTATION,
    "use_cache": True, "logits_to_keep": 1, "do_sample": False,
    "quantization": "NF4 double quantization", "compute_dtype": str(compute_dtype),
    "versions": {p: version(p) for p in ("torch", "transformers", "accelerate", "bitsandbytes")},
}
results = {"metadata": metadata, "answers": []}
if OUTPUT_JSON.exists():
    try:
        OUTPUT_JSON.unlink()
    except Exception:
        pass

done = {str(r["id"]) for r in results["answers"] if r.get("status") == "ok"}

def save():
    temporary = OUTPUT_JSON.with_suffix(".tmp")
    temporary.write_text(json.dumps(results, ensure_ascii=False, indent=2), encoding="utf-8")
    temporary.replace(OUTPUT_JSON)

@torch.no_grad()
def infer(messages):
    inputs = outputs = None
    # Força a liberação de qualquer fragmentação de memória antes de alocar os tensores do prompt
    gc.collect()
    torch.cuda.empty_cache()
    try:
        inputs = tokenizer.apply_chat_template(
            messages, add_generation_prompt=True, return_tensors="pt", return_dict=True,
        ).to(model.device)
        input_length = inputs["input_ids"].shape[1]
        if input_length + MAX_NEW_TOKENS > model.config.max_position_embeddings:
            raise ValueError("Prompt plus generation exceeds model window; no automatic truncation")
        
        with torch.inference_mode():
            outputs = model.generate(
                **inputs, do_sample=False, num_beams=1, max_new_tokens=MAX_NEW_TOKENS,
                use_cache=True, cache_implementation=CACHE_IMPLEMENTATION,
                logits_to_keep=1, return_dict_in_generate=False,
                output_scores=False, output_attentions=False, output_hidden_states=False,
                pad_token_id=tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id,
            )
        answer = tokenizer.decode(outputs[0, input_length:].cpu(), skip_special_tokens=True).strip()
        return {"status": "ok", "answer": answer, "input_tokens": input_length,
                "output_tokens": outputs.shape[1]-input_length}
    finally:
        del inputs, outputs
        gc.collect()
        torch.cuda.empty_cache()

for question in payload:
    if str(question["id"]) in done:
        continue
    
    gc.collect()
    torch.cuda.empty_cache()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    
    try:
        record = infer(question["prompt"])
    except torch.cuda.OutOfMemoryError:
        # Limpa o estado de erro do CUDA
        gc.collect()
        torch.cuda.empty_cache()
        record = {"status": "oom", "answer": None,
                  "error": "CUDA OOM; prompt preserved. Retry after restarting runtime or changing memory settings."}
    
    record.update(id=question["id"], expected_answer=question["expected_answer"],
                  peak_allocated_gib=torch.cuda.max_memory_allocated()/2**30,
                  peak_reserved_gib=torch.cuda.max_memory_reserved()/2**30)
    results["answers"].append(record)
    save()
    print(question["id"], record["status"], f"{record['peak_allocated_gib']:.2f} GiB", record.get("answer")[:100] if record.get("answer") else None)